In [2]:
import pandas as pd
import re

data = pd.read_csv("spelling_corpus.csv")

print(data.head())

if "correct_word" in data.columns:
    vocabulary = set(
        data["correct_word"]
        .dropna()
        .astype(str)
        .str.lower()
    )
else:
    vocabulary = set()

print("\nVocabulary Size:", len(vocabulary))
def edit_distance(word1, word2):

    m = len(word1)
    n = len(word2)

    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if word1[i - 1] == word2[j - 1]:
                cost = 0
            else:
                cost = 1

            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + cost
            )

    return dp[m][n]
def correct_word(word):
    if word.lower() in vocabulary:
        return word, 0

    candidates = []

    for candidate in vocabulary:

        distance = edit_distance(
            word.lower(),
            candidate
        )

        candidates.append(
            (candidate, distance)
        )

    candidates.sort(key=lambda x: x[1])

    if candidates:
        return candidates[0]

    return word, 0


def correct_query(query):

    words = re.findall(r"[a-zA-Z]+", query.lower())

    corrected_words = []
    errors = []

    for word in words:

        if word in vocabulary:

            corrected_words.append(word)

        else:

            suggestion, distance = correct_word(word)

            errors.append(
                (word, suggestion, distance)
            )

            corrected_words.append(suggestion)

    corrected_query = " ".join(corrected_words)

    return errors, corrected_query


query = input("\nEnter your search query: ")

errors, corrected_query = correct_query(query)

print("\nOriginal Query:")
print(query)

print("\nIncorrect Words and Suggested Corrections:")

if errors:

    for word, suggestion, distance in errors:

        print(
            word,
            "->",
            suggestion,
            "(Edit Distance:",
            distance,
            ")"
        )

else:

    print("No spelling errors found.")

print("\nCorrected Query:")
print(corrected_query)

  correct_word misspelled_word
0       Albert              Ab
1      America         Ameraca
2      America         Amercia
3     American        Ameracan
4        April           Apirl

Vocabulary Size: 6130



Enter your search query:  machne lerning cours



Original Query:
machne lerning cours

Incorrect Words and Suggested Corrections:
machne -> machine (Edit Distance: 1 )
lerning -> learning (Edit Distance: 1 )
cours -> court (Edit Distance: 1 )

Corrected Query:
machine learning court
